# Full and sliding-window attention

Which keys can each query attend to?

| Source | Local : full layers | Local window |
| --- | --- | --- |
| [Gemma 2](https://arxiv.org/abs/2408.00118) | 1:1 | 4096 |
| [Gemma 3, §5.2](https://arxiv.org/html/2503.19786v1#S5.SS2) | 5:1 | 1024 |
| Our starting config | 1:1 (four layers) | 64 |

Gemma 3 reports little perplexity change across its tested ratios. Our configs
start with 1:1, then 3:1, keeping the existing four layers and context 256.

Implementation: `models/attention.py` and `models/attention_mlx.py`. Window W
includes self: query i can see keys `max(0, i-W+1)` through `i`. The tiled
implementation computes only nearby score blocks. Learned positions and
multi-head attention remain as before; this is not the Gemma architecture.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

length = 16
window = 4
positions = np.arange(length)
distance = positions[:, None] - positions[None, :]
full = distance >= 0
local = full & (distance < window)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, mask, title in zip(axes, (full, local), ("Full causal", f"Sliding window: {window}")):
    ax.imshow(mask, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    ax.set(title=title, xlabel="Key position", ylabel="Query position")
fig.tight_layout()
plt.show()


In [ ]:
# Input-to-output layer order; edit the per-layer list in the model config.
layouts = {
    "full": ["full"] * 4,
    "1:1": ["sliding_window", "full", "sliding_window", "full"],
    "3:1": ["sliding_window", "sliding_window", "sliding_window", "full"],
}
for name, layout in layouts.items():
    print(name, " → ".join(layout))


Run either experiment from the repository root after preparing the sequential cache:

```sh
python -m llm_training.pretraining_mlx --config configs/pretraining/sequential_gpt_mlx_swa_1to1.toml
python -m llm_training.pretraining_mlx --config configs/pretraining/sequential_gpt_mlx_swa_3to1.toml
```

The configs keep the same data, token budget, width, depth, and training settings
as the full-attention sequential baseline. Use notebook 03 to compare losses.
